In [1]:
import pandas as pd
from pathlib import Path
from typing import List

/global/homes/b/brookluo/.local/perlmutter/pytorch2.0.1/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
import numpy as np

In [3]:
from astropy.table import Table

In [4]:
import sys
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

In [5]:
from src.util import get_info_from_html, make_webpage

In [6]:
vidir = Path("/global/u1/b/brookluo/decam-exposure-quality/data/vi")

In [7]:
alexdir = vidir / "alex"
frankdir = vidir / "frank"
yldir = vidir / "yl"

In [8]:
datadir = Path("/pscratch/sd/b/brookluo/decam-exposure/dr11/data/bad_candidates")
mydr11 = Path("/pscratch/sd/b/brookluo/decam-exposure/dr11/data")

In [9]:
dr9_dir = Path('/global/cfs/cdirs/cosmo/work/legacysurvey/dr9')
dr8_dir = Path('/global/cfs/cdirs/cosmo/work/legacysurvey/dr8')
dr10_dir = Path('/global/cfs/cdirs/cosmo/work/legacysurvey/dr10')
# dr10_tab = Table.read(dr10_dir / "survey-ccds-decam-dr10.fits.gz")
dr11_dir = Path('/global/cfs/cdirs/cosmo/work/legacysurvey/dr11')

In [10]:
dr11tab = Table.read("/pscratch/sd/b/brookluo/decam-exposure/dr11/data/dr11-suvey-ccds.fits")

In [11]:
alldfs = []
for csv in datadir.glob("*.csv"):
    alldfs.append(pd.read_csv(csv))
alldf = pd.concat(alldfs, ignore_index=True)

In [12]:
alexdfs = []
for fp in alexdir.glob("*.xlsx"):
    alexdfs.append(pd.read_excel(fp))
alexdf = pd.concat(alexdfs, ignore_index=True).rename(columns={"Is_bad?": "Is_bad"})

In [13]:
frankdfs = []
for fp in frankdir.glob("*.xlsx"):
    df = pd.read_excel(fp)
    if df.columns[0] != "expnum":
        df = pd.read_excel(fp, header=1)
    frankdfs.append(df)
    # print(df.columns)
    # print(len(df.Reason[~df["Is_bad?"].isna()]))
frankdf = pd.concat(frankdfs, ignore_index=True).rename(columns={"Is_bad?": "Is_bad"})

In [14]:
yldfs = []
for fp in yldir.glob("*.xlsx"):
    df = pd.read_excel(fp)
    if df.columns[0] != "expnum":
        df = pd.read_excel(fp, header=1)
    yldfs.append(df)
yldf = pd.concat(yldfs, ignore_index=True).rename(columns={"Is_bad?": "Is_bad"})

In [15]:
yldf_not_nan = yldf[~yldf["Is_bad"].isna()]

In [16]:
sum(yldf_not_nan["Is_bad"].isin(['y', 'y?', 'yes', 'yes?']))

467

In [17]:
find_uni = lambda onedf: np.unique(onedf[~onedf["Is_bad"].isna()]["Is_bad"])
print("YL response:", find_uni(yldf))
print("Frank response:", find_uni(frankdf))
print("Alex response:", find_uni(alexdf))

YL response: ['n' 'n?' 'no' 'y' 'y?' 'yes' 'yes?']
Frank response: [0. 1.]
Alex response: ['maybe' 'no' 'yes' 'yes ']


In [18]:
def merge_vi_info(masterdf, vidf, viname, bad_symbols: List[str]):
    # if reason is None, that means the expnum is not VIed by that person
    badexp = np.zeros(len(masterdf), dtype=bool)
    badrea = np.full(len(masterdf), dtype=object, fill_value=None)
    reason = []
    vidf_not_nan = vidf[~vidf["Is_bad"].isna()]
    # now use response dictonary to parse the output where the bad is true
    # bad_symbol = resp_dict["bad_symbol"]
    vidf_bad = vidf_not_nan[vidf_not_nan["Is_bad"].isin(bad_symbols)]
    _, idx1, idx2 = np.intersect1d(masterdf.expnum, vidf_bad.expnum,
                                   return_indices=True)
    badexp[idx1] = True
    badrea[idx1] = vidf_bad.iloc[idx2].Reason
    masterdf[f"{viname}_badexp"] = badexp
    masterdf[f"{viname}_reason"] = badrea

In [19]:
merge_vi_info(alldf, alexdf, "alex", ['maybe', "yes", 'yes '])
merge_vi_info(alldf, frankdf, "frank", [1.])
merge_vi_info(alldf, yldf, "yufeng", ['y', 'y?', 'yes', 'yes?'])

In [20]:
at_least_one = alldf["alex_badexp"] | alldf["frank_badexp"] | alldf["yufeng_badexp"]
two_agree = (alldf["alex_badexp"] & alldf["frank_badexp"]) \
            | (alldf["alex_badexp"] & alldf["yufeng_badexp"]) \
            | (alldf["frank_badexp"] & alldf["yufeng_badexp"])

all_agree = alldf["alex_badexp"] & alldf["frank_badexp"] & alldf["yufeng_badexp"]

In [21]:
len(alldf[at_least_one]) / len(alldf)

0.25127107305325125

In [22]:
len(alldf)

3737

In [23]:
sum(at_least_one)

939

In [45]:
badexp = alldf[at_least_one]

In [46]:
badexp = badexp.sort_values(by=['expnum'])

In [47]:
matched_exp, badexp_idx, dr11_idx = np.intersect1d(badexp["expnum"], dr11tab["expnum"], return_indices=True)

In [48]:
np.unique(dr11tab[dr11_idx]["plver"])

V4.10
V4.8.1
V4.8.2
V4.8.2a
V4.9
V5.0
V5.2
V5.2.1
V5.2.2
V5.2.2LS
V5.2.3


In [27]:
plver = np.full(len(badexp), fill_value=None)
plver[badexp_idx] = dr11tab[dr11_idx]["plver"]
plver = plver.astype(str)

In [28]:
badexp.insert(2, "plver", plver)

In [29]:
sum(plver == "V4.8.1")

15

In [30]:
html_dir = Path("/global/cfs/cdirs/desicollab/users/brookluo/decam-exposure-data/dr11")

In [31]:
alldf.columns

Index(['expnum', 'image_fname', 'ML_reason', 'alex_badexp', 'alex_reason',
       'frank_badexp', 'frank_reason', 'yufeng_badexp', 'yufeng_reason'],
      dtype='object')

In [32]:
final_bad_exp_html = []
for html in html_dir.glob(r"?_[low,high]*.html"):
    entry = get_info_from_html(html)
    for et in entry:
        idx = np.where(badexp["expnum"] == int(et[1]))[0]
        if len(idx) > 0:
            # print("here")
            onerow = badexp.iloc[idx[0]]
            et.insert(-1, f"plver: {onerow['plver']}")
            # print(onerow)
            vi = [name for name in ("alex", "frank", "yufeng") if onerow[f"{name}_badexp"]]
            et.insert(-1, f"confirmed by: {(',').join(vi)}")
            final_bad_exp_html.append(et)
        # break

expnum = [int(et[1]) for et in final_bad_exp_html]
sort_idx = np.argsort(expnum)

final_bad_exp_html = [final_bad_exp_html[i] for i in sort_idx]

In [55]:
# def make_webpage(master_list, pack_idx, root_dir, base_name, num_element=400):
#     count = 0
#     base_tmpl = "<table>{}\n</table>"
#     content = []
#     start_exp = -1
#     for i, idx in enumerate(pack_idx):
#         if start_exp == -1:
#             start_exp = master_list[idx][1]
#         content.append("<br>".join(master_list[idx]))
#         if num_element > 0 and i and i % num_element == 0:
#             end_exp = master_list[idx][1]
#             with open(root_dir / f"{count}_{base_name}_{start_exp}_{end_exp}.html", "w") as web:
#                 web.write(base_tmpl.format("\n".join(content)))
#             count += 1
#             start_exp = -1
#             content = []
#     if len(content):
#         end_exp = master_list[idx][1]
#         with open(root_dir / f"{count}_{base_name}_{start_exp}_{end_exp}.html", "w") as web:
#             web.write(base_tmpl.format("\n".join(content))) 

# make_webpage(final_bad_exp_html, np.arange(len(final_bad_exp_html)), html_dir, "vi_checked", 1e4)

In [34]:
len(final_bad_exp_html)

939

In [35]:
filters = np.array([fn.split("_")[-2] for fn in alldf["image_fname"]])

In [36]:
sum(filters == "Y")

331

In [37]:
from astropy.table import Table

In [38]:
tab = Table.from_pandas(badexp)

In [54]:
mydr11

PosixPath('/pscratch/sd/b/brookluo/decam-exposure/dr11/data')

In [51]:
badexp.to_csv(mydr11 / "dr11_final_selected_bad_exposures.csv", index=False)

In [52]:
alldf

,expnum,image_fname,ML_reason,alex_badexp,alex_reason,frank_badexp,frank_reason,yufeng_badexp,yufeng_reason
0,135574,c4d_120923_034339_ooi_g_ls11,PSF,False,None,False,None,False,None
1,137380,c4d_121001_062046_ooi_g_ls11,NObjects,True,something in the flat? pretty weird,False,None,True,transparency
2,137389,c4d_121001_064632_ooi_z_ls11,NObjects,True,NaN,False,None,True,transparency
3,137399,c4d_121001_070453_ooi_g_ls11,Clouds_transparency,True,NaN,False,None,True,transparency
4,138007,c4d_121003_030521_ooi_r_ls11,Out_of_focus,True,saturation,False,None,True,bad ccd
...,...,...,...,...,...,...,...,...,...
3732,929824,c4d_200204_041315_ooi_i_v1,Telescope_Moving,False,None,False,None,False,None
3733,929828,c4d_200204_042158_ooi_i_v1,Ghost_Scatter,False,None,False,None,False,None
3734,930303,c4d_200205_071902_ooi_g_v1,Telescope_Moving,True,telescop moving,False,None,False,None
3735,930823,c4d_200207_022732_ooi_i_v1,Ghost_Scatter,False,None,False,None,False,None


In [53]:
badexp

,expnum,image_fname,ML_reason,alex_badexp,alex_reason,frank_badexp,frank_reason,yufeng_badexp,yufeng_reason
1,137380,c4d_121001_062046_ooi_g_ls11,NObjects,True,something in the flat? pretty weird,False,None,True,transparency
2,137389,c4d_121001_064632_ooi_z_ls11,NObjects,True,NaN,False,None,True,transparency
3,137399,c4d_121001_070453_ooi_g_ls11,Clouds_transparency,True,NaN,False,None,True,transparency
4,138007,c4d_121003_030521_ooi_r_ls11,Out_of_focus,True,saturation,False,None,True,bad ccd
5,138035,c4d_121003_034911_ooi_r_ls11,Out_of_focus,True,saturation,False,None,True,bad ccd
...,...,...,...,...,...,...,...,...,...
3100,1263170,c4d_231230_031502_ooi_i_v1,"NObjects,Wonky",True,transparency,True,transparency/moon,False,None
3331,1263181,c4d_231230_033820_ooi_i_v1,NObjects,False,None,False,None,True,scattered light
3332,1266993,c4d_240112_072903_ooi_r_ls11,Telescope_Moving,False,None,False,None,True,ghost
3335,1267847,c4d_240115_074256_ooi_g_ls11,Ghost_Scatter,False,None,False,None,True,trail
